# KI（knowledge insulation）：断在哪两个矩阵上对应文章「π 系列具身模型（03）」。KI 的操作只有一句话：**在 action expert 读 backbone的 K / V 时加 stop-gradient**。这句话里有两个容易被略过的细节，本 notebook 就是把它们钉死：1. **只断两个矩阵** $$\mathbf{K}_b,\mathbf{V}_b$$，不是四个。$$\mathbf{Q}_b$$ 和输出投影   为什么不用管——因为它们本来就拿不到流匹配的梯度（掩码决定了 backbone 行读不到 action token）。2. **断了之后 $$lpha$$ 就不是旋钮了**。流匹配损失与离散动作损失作用在**互不相交**的   两组权重上，所以它们的相对权重不会影响 backbone 的轨迹（实测：逐位相同）。只验机制，不重算论文成绩（LIBERO / DROID / 7.5 倍步数都是引用）。全部跑完输出**断言总数 = 40**。> notebook 由同目录的 `ki_lib.py` + `ki_lab.py` 按函数边界自动生成并逐格执行，> 两份 .py 是唯一事实来源；直接 `python3 ki_lab.py` 也能得到完全一样的结果> （见 `results/stdout.txt`）。

In [1]:
"""KI（knowledge insulation）的可复现内核：一个 2x2 块注意力 + 手写反向。对应文章：π 系列具身模型（03）KI —— 断在哪两个矩阵上。为什么要手写反向：本实验要验证的核心事实是 **stop-gradient 之后某个梯度恒等于 0**。有限差分只测「前向函数对参数的敏感度」，而 stop-gradient 在前向里是恒等映射——用有限差分去测它，测出来的是「如果不断梯度会是多少」，永远不为 0。所以只能手写反向，再用有限差分（在 sg 关闭时）去校准这份手写反向。结构（对齐 π0 / KI 论文的 2x2 块分解）：    P = ( P_bb    0  )      E = ( P_bb · V_b                        )        ( P_ab  P_aa )          ( P_ab · sg(V_b) + P_aa · V_a       )token 分三块：    O —— 观测与指令（4 个物体 token + 1 个指令 token + 1 个回答 token），backbone    R —— AR 动作 token 的槽位（5 个），backbone，KI 论文里 FAST token 就落在这儿    A —— action expert 的连续动作 token（5 个）掩码：O 双向；R 看 O + R；A 看 O + A，**A 看不见 R**（论文原文：两类动作表示互不 attend）；任何 backbone token 都看不见 A（π0 的掩码已经如此）。"""import numpy as np

## 1. 尺寸与注意力掩码token 分三块，对应 π0 / KI 论文的 2×2 块分解：| 块 | 位置 | 内容 | 谁 || --- | --- | --- | --- || O | `0:N_O` | 4 个图像 token + 指令 token + 回答 token | backbone || R | `N_O:N_B` | 离散动作（FAST）token 的槽位 | backbone || A | `N_B:` | action expert 的连续动作 token | expert |掩码三条规则：O 内部双向；R 看 O + R；**A 看 O + A，但看不见 R**（论文原文：两类动作表示互不 attend）；任何 backbone token 都看不见 A。

In [2]:
# ------------------------------------------------------------------ 尺寸D_B, D_A, H = 32, 16, 8          # backbone 宽 / expert 宽 / 单头 head_dim（可不同宽、同 head_dim）N_OBJ, N_R, N_A, D_ACT = 4, 5, 5, 7N_O = N_OBJ + 2                  # 4 图像 + 指令 + 答案N_B = N_O + N_R                  # backbone token 数（含 AR 槽位）N_ALL = N_B + N_APOS_LANG, POS_ANS = 4, 5         # O 内部的指令位与回答位V, C_LANG = 16, 4                # 动作 token 词表 / 语言任务的类别数R_A = 3                          # 动作真正依赖的读数个数（A 的秩）R_SLICE = slice(N_O, N_B)A_SLICE = slice(N_B, N_ALL)A_ROWS = np.zeros(N_ALL, dtype=bool)A_ROWS[N_B:] = TrueNON_A_ROWS = ~A_ROWSdef build_mask():    """注意力掩码。行 = 查询位置，列 = 被 attend 的位置。"""    m = np.zeros((N_ALL, N_ALL), dtype=bool)    O = slice(0, N_O)    m[O, O] = True                       # 观测块内部双向    m[R_SLICE, O] = True                 # AR 槽位看得见观测    m[R_SLICE, R_SLICE] = True           # AR 槽位之间双向    m[A_SLICE, O] = True                 # expert 看得见观测（读 K_b / V_b 的地方）    m[A_SLICE, A_SLICE] = True           # expert 内部双向    # 刻意留空：A 看不见 R（两类动作表示互不 attend）；backbone 永远看不见 A    return mMASK = build_mask()BIG_NEG = -1e9# ------------------------------------------------------------------ 数据

## 2. 合成世界一个样本 = 4 个物体（名字嵌入 `NAME[j]` + 当次外观扰动 `eps`）+ 一条指令（指定 `c`）。* **语言标签**只由被选中物体的当次扰动决定：$$rg\max(\epsilon_c \mathbf{P}_{gt}^{\top})$$。  关键是它**不依赖 NAME**——否则回答位自带的 `NAME[c]` 就是泄漏，打乱图像也打不下去  （第一版就是这样，负对照只掉到 0.60）。* **动作块**依赖**整幅场景**（4 个物体扰动的均值）在 $$\mathbf{U}$$ 上的 $$R_A=3$$ 个读数，  且 $$\mathbf{U}\perp \mathbf{P}_{gt}$$：语言与动作需要的特征正交，这正是秩 8 的 K/V 瓶颈  能卡住的地方。不是「选中的那一个」，因为 expert 的 query 不含指令，注意力只能摊在  4 个物体上，选中那一个的信号会被另外 3 个的独立噪声稀释 4 倍（实测 act 全卡在 0.52）。

In [3]:
class Scene:    """一次前向需要的全部合成数据。物体向量 -> 真值动作块 / 语言标签 / 离散动作 token。"""    def __init__(self, seed, n, noise=0.6):        rng = np.random.default_rng(seed)        # 固定的「世界」参数（真值生成器），所有 recipe 共用。        # P_gt 与 NAME 都取**正交**行：随机高斯行会让 4 个类别的 argmax 区域互相纠缠，        # 连「在精确特征上跑多项 logistic」都只能到 0.85（实测），语言任务的天花板就被        # 标签本身的几何卡住了，看不出配方差异。正交行把天花板抬到 0.97（实测）。        Qg, _ = np.linalg.qr(rng.normal(size=(D_B, D_B)))        self.P_gt = Qg[:, :C_LANG].T        # 动作只依赖「语言任务用不到的那部分特征」：把 W_gt 的行空间投影到 P_gt 行空间的正交补上。        # 这是「VLM 预训练里没有机器人数据、冻住的表示不够用」的合成对应物。        Qp, _ = np.linalg.qr(self.P_gt.T)                 # (D_B, C_LANG)        # 动作只依赖 R_A 个标量读数（U 的行），且 U ⊥ P_gt 的行空间：        # 语言任务与动作任务需要的特征正交，这正是 rank-H 的 K/V 瓶颈能卡住的地方        U = rng.normal(size=(R_A, D_B))        U = U - (U @ Qp) @ Qp.T          # 投到 P_gt 行空间的正交补上        Uq, _ = np.linalg.qr(U.T)        # 再正交化（投影后各行已不互相正交，        self.U = Uq.T                    # 不补这一步 Cov(s) ≠ σ_s²·I，Σ_A 就配不准）        # Vg 取正交阵的前 R_A 列：这样 s @ Vg 在 35 个坐标上的方差**处处相等**        # （正交阵的行与列都是单位范数）。用 iid 高斯的话各维方差会按 chi^2_3 波动，        # 低方差维上 A 变弱，u 就能从 x_t 里直接反解出来（见下面 UNPRED 那段）。        Qv, _ = np.linalg.qr(rng.normal(size=(N_A * D_ACT, N_A * D_ACT)))        self.Vg = Qv[:, :R_A].T        # 物体的「名字」嵌入：跨样本一致，视觉编码器把图像映射到与它同一个语义空间，        # 这是 VLM 预训练的基本假设，也是「按指令挑物体」能成立的前提        Qn, _ = np.linalg.qr(rng.normal(size=(D_B, D_B)))        self.NAME = Qn[:, :N_OBJ].T        self.ans_tok = rng.normal(size=(D_B,)) / np.sqrt(D_B)        self.Slot_emb = rng.normal(size=(N_R, D_B)) / np.sqrt(D_B)        self.Wa_in = rng.normal(size=(D_ACT + 1, D_A)) / np.sqrt(D_ACT + 1)        self.w_bin = rng.normal(size=(D_ACT,))        self.w_bin /= np.linalg.norm(self.w_bin)        self.n = n        # 每个物体 = 名字嵌入 + 当次的外观扰动        self.eps = noise * rng.normal(size=(n, N_OBJ, D_B)) / np.sqrt(D_B)        self.obj = self.NAME[None, :, :] + self.eps        self.c = rng.integers(0, N_OBJ, size=n)                      # 指令挑中的那个物体        # 动作块只由「当次实例的外观扰动」决定：这部分信息语言任务用不到（U ⊥ P_gt），        # 所以预训练过的 backbone 在冻住时把它压掉了 —— 这正是论文里「冻住不行」的合成对应物        picked_eps = self.eps[np.arange(n), self.c]        # 动作依赖**整幅场景**（4 个物体扰动的均值），语言依赖**指令挑中的那一个**。        # 这不是随便选的：expert 的 query 只来自带噪动作与时间，不含指令，        # 注意力只能摊在 4 个物体上；若动作只依赖被选中那个，信号会被另外 3 个的        # 独立噪声稀释 4 倍（实测 act 全卡在 0.52，与「不看 backbone」的 0.523 一致）。        self.s = self.eps.mean(axis=1) @ self.U.T                     # (n, R_A) 动作真正依赖的读数        A_sig = (self.s @ self.Vg).reshape(n, N_A, D_ACT)             # 可预测部分（秩 R_A）        # 不可预测部分：观测里根本没有的信息，谁也学不会（对应真实机器人里        # 「同一场景下两次示范也不一样」）。它必须存在，否则 A 严格低秩，        # 秩之外的方向上 A≡0，那些方向的 u 可以直接从 x_t 反解出来。        self.unpred = 0.5                                            # 噪声标准差 / 信号标准差        A_raw = A_sig + self.unpred * A_sig.std() * rng.normal(size=A_sig.shape)        self.a_scale = float(1.0 / A_raw.std())                      # 归一化到 std≈1        self.A = A_raw * self.a_scale        # 流匹配的噪声 ω 取「与 A 同协方差」的高斯，而不是标准高斯。        # 原因：x_t = τA + (1−τ)ω 里本来就带着 A 的信息，expert 哪怕完全不读 backbone        # 也能反解出一部分 u —— 实测这会让 freeze 与 ki 的动作误差一样（0.428 / 0.429），        # 指标彻底失去分辨力。cov(u, x_t) = (1−τ)Σ_ω − τΣ_A，令 Σ_ω = Σ_A 则在 τ=0.5 处        # 严格为 0；A 与 ω 都是高斯，不相关即独立，捷径被完全堵死。        # 代价：τ≠0.5 时仍有 (1−2τ) 量级的残余相关（τ→0 或 1 时最多白送 50%），        # 所以评估一律钉在 τ=0.5（`sample_batch(fixed_tau=0.5)`）。        # Σ_A = a_scale²(σ_s²·Π + (u·σ_A)²·I)，Π = Vg^T Vg 是秩 R_A 的投影。        # 用闭式而不是经验协方差：A 是高斯的，协方差对得上就是严格独立。        Cs = np.cov(self.s.T)                                        # 用样本协方差，避开抽样涨落        Sig = self.a_scale ** 2 * (self.Vg.T @ Cs @ self.Vg                                   + (self.unpred * A_sig.std()) ** 2 * np.eye(N_A * D_ACT))        self.omega_L = np.linalg.cholesky(Sig)                        # Σ_A^{1/2}（35×35）        # 语言标签 = 被选中物体「当次外观」在 P_gt 上的读数取 argmax。        # 关键：标签只依赖 eps，不依赖 NAME —— 否则回答位自带的 NAME[c] 就是泄漏，        # 模型不用看图也能猜对（负对照会打不下去）。同时在 x_b 里 NAME[c] 只作为        # 「指令」出现在查询侧，模型必须靠它去 4 个物体里挑出对应的那个，        # 这才是「语言跟随」在合成任务里的最小形式。        self.y_lang = np.argmax(picked_eps @ self.P_gt.T, axis=-1)        # 离散动作 token：把每个动作步沿固定方向投影后分箱（就是 02 篇讲的 naive binning）        z = self.A @ self.w_bin                                      # (n, N_A)        lo, hi = z.min(), z.max()        edges = lo + (hi - lo) * np.arange(1, V) / V        self.y_ar = np.clip(np.searchsorted(edges, z), 0, V - 1).astype(np.int64)        self.edges = edges    def x_b(self, idx):        """backbone 的输入 token（嵌入层冻结，等价于预训练好的 SigLIP + 词嵌入）。"""        n = len(idx)        X = np.zeros((n, N_B, D_B))        X[:, :N_OBJ, :] = self.obj[idx]                              # 4 个图像 token        X[:, POS_LANG, :] = self.NAME[self.c[idx]]                   # 指令 token（= 名字嵌入）        X[:, POS_ANS, :] = self.NAME[self.c[idx]] + 0.5 * self.ans_tok  # 回答位：带着指令        X[:, R_SLICE, :] = self.Slot_emb[None, :, :]                 # AR 槽位        return X    def x_a(self, idx, tau, omega):        """expert 的输入：[加噪动作, 流时间] 过一层固定投影。"""        A = self.A[idx]        x_t = tau[:, None, None] * A + (1.0 - tau[:, None, None]) * omega        t_col = np.broadcast_to(tau[:, None, None], (x_t.shape[0], N_A, 1))        raw = np.concatenate([x_t, t_col], axis=-1)          # (n, N_A, D_ACT+1)        return np.tanh(raw @ self.Wa_in), x_t, A

### 抽一批：流匹配的噪声要与动作块同协方差这是整个实验里最要紧的一处构造。`x_t = τA + (1−τ)ω` 里本来就带着 $$A$$ 的信息，expert 哪怕完全不读 backbone 也能反解出一部分 $$u$$ —— 第一版用标准高斯噪声，结果`freeze` 与 `ki` 的动作误差一样（0.428 / 0.429），指标彻底失去分辨力。$$\operatorname{cov}(u, x_t) = (1-τ)\Sigma_ω - τ\Sigma_A$$。令 $$\Sigma_ω=\Sigma_A$$，在 τ=0.5 处严格为 0；$$A$$ 与 $$ω$$ 都是高斯，不相关即独立，捷径被完全堵死。所以评估一律钉在 τ=0.5。

In [4]:
def sample_batch(sc, idx, rng, fixed_tau=None):    """抽一批，返回 forward 需要的全部量。fixed_tau 用于评估时钉住噪声水平。"""    n = len(idx)    if fixed_tau is None:        tau = rng.beta(1.5, 1.0, size=n) * 0.999 + 0.001    else:        tau = np.full(n, fixed_tau)    z = rng.normal(size=(n, N_A * D_ACT))    omega = (z @ sc.omega_L.T).reshape(n, N_A, D_ACT)          # ω ~ N(0, Σ_A)    X_b = sc.x_b(idx)    X_a, x_t, A = sc.x_a(idx, tau, omega)    return dict(        X_b=X_b, X_a=X_a, tau=tau, u=omega - A, x_t=x_t,        y_lang=sc.y_lang[idx], y_ar=sc.y_ar[idx], A=A,        # 预训练阶段给 AR 头用的「语言类」目标：让离散头像真实系统那样从预训练的输出层起步        y_ar_lang=np.broadcast_to(sc.y_lang[idx][:, None], (len(idx), N_R)),    )

## 3. 参数`BONE` 六项是 backbone 侧，`EXPERT` 五项是 expert 侧。论文说的「断梯度」只发生在expert 读 backbone 的 K / V 这两条通路上，两组权重本身都照常训练。

In [5]:
# ------------------------------------------------------------------ 参数def init_params(rng):    def w(shape):        return rng.normal(size=shape) / np.sqrt(shape[0])    return dict(        Wq_b=w((D_B, H)), Wk_b=w((D_B, H)), Wv_b=w((D_B, H)), Wo_b=w((H, D_B)),        Wq_a=w((D_A, H)), Wk_a=w((D_A, H)), Wv_a=w((D_A, H)), Wo_a=w((H, D_A)),        W_lang=w((D_B, C_LANG)), W_ar=w((D_B, V)), W_flow=w((D_A, D_ACT)),    )BONE = ["Wq_b", "Wk_b", "Wv_b", "Wo_b", "W_lang", "W_ar"]      # backbone 侧EXPERT = ["Wq_a", "Wk_a", "Wv_a", "Wo_a", "W_flow"]            # expert 侧ALL_P = BONE + EXPERTdef zeros_grad(P):    return {k: np.zeros_like(v) for k, v in P.items()}

## 4. 前向：stop-gradient 落在哪按论文的 2×2 块分解：$$P = \begin{pmatrix} P_{bb} & 0 \\ P_{ab} & P_{aa} \end{pmatrix},\qquadE = \begin{pmatrix} P_{bb} V_b \\ P_{ab}\, \mathrm{sg}(V_b) + P_{aa} V_a \end{pmatrix}$$$$P_{ab}$$ 那一项的 logits 也走 $$\mathrm{sg}(K_b)$$。实现上只有 **A 行**用 sg 版（backbone 行本来就不看 expert token，用不用都一样），前向数值逐位不变——这也是为什么「断梯度不改变前向」可以直接断言成 `== 0.0`。

In [6]:
# ------------------------------------------------------------------ 前向def forward(P, b, sg):    """sg=True 时对 P_ab / V_b 的跨模块通路做 stop-gradient。返回 (输出, cache)。"""    X_b, X_a = b["X_b"], b["X_a"]    q_b = X_b @ P["Wq_b"]    k_b = X_b @ P["Wk_b"]    v_b = X_b @ P["Wv_b"]    q_a = X_a @ P["Wq_a"]    k_a = X_a @ P["Wk_a"]    v_a = X_a @ P["Wv_a"]    q = np.concatenate([q_b, q_a], axis=1)    k = np.concatenate([k_b, k_a], axis=1)    v = np.concatenate([v_b, v_a], axis=1)    scale = H ** -0.5    logits = np.einsum("bth,bsh->bts", q, k) * scale    if sg:        # 只有 A 行会用到「断掉 backbone 梯度」的那一版 logits        k_sg = np.concatenate([k_b, k_a], axis=1)          # 前向值与 k 相同        logits_sg = np.einsum("bth,bsh->bts", q, k_sg) * scale        logits = np.where(A_ROWS[None, :, None], logits_sg, logits)    else:        k_sg = None        logits_sg = None    masked = np.where(MASK[None, :, :], logits, BIG_NEG)    probs = _softmax(masked)    enc_full = np.einsum("bts,bsh->bth", probs, v)    if sg:        v_sg = np.concatenate([v_b, v_a], axis=1)          # 前向值与 v 相同        enc_sg = np.einsum("bts,bsh->bth", probs, v_sg)        enc = np.where(A_ROWS[None, :, None], enc_sg, enc_full)    else:        enc_sg, v_sg = None, None        enc = enc_full    # 两套输出投影（宽度不同，所以各回各的宽）    out_b = enc[:, :N_B, :] @ P["Wo_b"]    out_a = enc[:, N_B:, :] @ P["Wo_a"]    H_b = X_b + out_b                       # 残差    H_a = X_a + out_a    out = dict(        logits_lang=H_b[:, POS_ANS, :] @ P["W_lang"],        logits_ar=H_b[:, R_SLICE, :] @ P["W_ar"],        pred_flow=H_a @ P["W_flow"],    )    cache = dict(q=q, k=k, v=v, k_sg=k_sg, v_sg=v_sg, probs=probs, enc=enc,                 enc_full=enc_full, enc_sg=enc_sg, H_b=H_b, H_a=H_a,                 X_b=X_b, X_a=X_a, sg=sg, out=out)    return out, cachedef _softmax(x):    x = x - x.max(axis=-1, keepdims=True)    e = np.exp(x)    return e / e.sum(axis=-1, keepdims=True)def ce_loss(logits, y):    """返回 (均值损失, dlogits)。"""    n = logits.shape[0]    z = logits - logits.max(axis=-1, keepdims=True)    p = _softmax(z)    m = logits.shape[1] if logits.ndim == 3 else 1    loss = -np.mean(np.log(np.take_along_axis(p, y[..., None], axis=-1) + 1e-12))    d = p.copy()    np.put_along_axis(d, y[..., None], np.take_along_axis(d, y[..., None], axis=-1) - 1.0, axis=-1)    d /= (n * (m if logits.ndim == 3 else 1))    return loss, ddef mse_loss(pred, target):    numel = pred.size    loss = np.mean(np.square(pred - target))    return loss, 2.0 * (pred - target) / numel

## 5. 手写反向必须手写：stop-gradient 在前向里是恒等映射，有限差分测的是「如果不断梯度会是多少」，永远不为 0，测不出本实验要验的那件事。写完再用有限差分（在 sg 关闭时）校准。

In [7]:
# ------------------------------------------------------------------ 反向def backward(P, b, cache, g):    """g: dict，各输出的上游梯度。返回 grads。"""    G = zeros_grad(P)    H_b, H_a = cache["H_b"], cache["H_a"]    X_b, X_a = cache["X_b"], cache["X_a"]    dH_b = np.zeros_like(H_b)    dH_a = np.zeros_like(H_a)    # ---- 三个头    dH_b[:, POS_ANS, :] += g["logits_lang"] @ P["W_lang"].T    G["W_lang"] += H_b[:, POS_ANS, :].T @ g["logits_lang"]    d_ar = g["logits_ar"]                                    # (n, N_R, V)    dH_b[:, R_SLICE, :] += d_ar @ P["W_ar"].T    G["W_ar"] += H_b[:, R_SLICE, :].reshape(-1, D_B).T @ d_ar.reshape(-1, V)    d_flow = g["pred_flow"]                                  # (n, N_A, D_ACT)    dH_a += d_flow @ P["W_flow"].T    G["W_flow"] += H_a.reshape(-1, D_A).T @ d_flow.reshape(-1, D_ACT)    # ---- 输出投影（残差：dout = dH）    enc_b, enc_a = cache["enc"][:, :N_B, :], cache["enc"][:, N_B:, :]    G["Wo_b"] += enc_b.reshape(-1, H).T @ dH_b.reshape(-1, D_B)    G["Wo_a"] += enc_a.reshape(-1, H).T @ dH_a.reshape(-1, D_A)    denc = np.zeros_like(cache["enc"])    denc[:, :N_B, :] += dH_b @ P["Wo_b"].T    denc[:, N_B:, :] += dH_a @ P["Wo_a"].T    # ---- enc 的两条支路：sg 时 A 行走 sg 版，其余走普通版；非 sg 时全走普通版    probs, v = cache["probs"], cache["v"]    sg = cache["sg"]    denc_full = np.where(A_ROWS[None, :, None], 0.0, denc) if sg else denc    dprobs = np.einsum("bth,bsh->bts", denc_full, v)    dv = np.einsum("bts,bth->bsh", probs, denc_full)    if sg:        denc_sg = np.where(A_ROWS[None, :, None], denc, 0.0)        dprobs += np.einsum("bth,bsh->bts", denc_sg, cache["v_sg"])        # sg(V_b) 这一支不给 V_b 任何梯度；只把 expert 自己的 V_a 接上        dv[:, N_B:, :] += np.einsum("bts,bth->bsh", probs, denc_sg)[:, N_B:, :]    # ---- softmax（probs 本身是用混合后的 logits 算的，所以这里只有一份）    dlogits = probs * (dprobs - np.sum(dprobs * probs, axis=-1, keepdims=True))    dlogits = np.where(MASK[None, :, :], dlogits, 0.0)    scale = H ** -0.5    dlogits_full = np.where(A_ROWS[None, :, None], 0.0, dlogits) if sg else dlogits    dq = np.einsum("bts,bsh->bth", dlogits_full, cache["k"]) * scale    dk = np.einsum("bts,bth->bsh", dlogits_full, cache["q"]) * scale    if sg:        dlogits_sg = np.where(A_ROWS[None, :, None], dlogits, 0.0)        dq += np.einsum("bts,bsh->bth", dlogits_sg, cache["k_sg"]) * scale        # sg(K_b) 这一支不给 K_b 任何梯度；只把 expert 自己的 K_a 接上        dk[:, N_B:, :] += np.einsum("bts,bth->bsh", dlogits_sg, cache["q"])[:, N_B:, :] * scale    # ---- Q/K/V 投影    G["Wq_b"] += X_b.reshape(-1, D_B).T @ dq[:, :N_B, :].reshape(-1, H)    G["Wq_a"] += X_a.reshape(-1, D_A).T @ dq[:, N_B:, :].reshape(-1, H)    G["Wk_b"] += X_b.reshape(-1, D_B).T @ dk[:, :N_B, :].reshape(-1, H)    G["Wk_a"] += X_a.reshape(-1, D_A).T @ dk[:, N_B:, :].reshape(-1, H)    G["Wv_b"] += X_b.reshape(-1, D_B).T @ dv[:, :N_B, :].reshape(-1, H)    G["Wv_a"] += X_a.reshape(-1, D_A).T @ dv[:, N_B:, :].reshape(-1, H)    return Gdef grads(P, b, sg, w_flow=1.0, w_ar=1.0, w_lang=0.0, ar_tgt=None):    """一步前向 + 反向，返回 (损失 dict, grads)。ar_tkg 可把 AR 目标换成预训练阶段的替代目标。"""    tgt = b["y_ar"] if ar_tgt is None else ar_tgt    out, cache = forward(P, b, sg)    losses = {}    g = {}    if w_lang > 0:        losses["lang"], g["logits_lang"] = ce_loss(out["logits_lang"], b["y_lang"])        g["logits_lang"] *= w_lang    else:        g["logits_lang"] = np.zeros_like(out["logits_lang"])    if w_ar > 0:        losses["ar"], g["logits_ar"] = ce_loss(out["logits_ar"], tgt)        g["logits_ar"] *= w_ar    else:        g["logits_ar"] = np.zeros_like(out["logits_ar"])    if w_flow > 0:        losses["flow"], g["pred_flow"] = mse_loss(out["pred_flow"], b["u"])        g["pred_flow"] *= w_flow    else:        g["pred_flow"] = np.zeros_like(out["pred_flow"])    return losses, backward(P, b, cache, g), out, cache

## 6. 优化器Adam 的逐参数归一化是「α 不影响结果」的另一半原因（另一半是断梯度本身），B 组会把这两半切开。SGD 只用于对照。

In [8]:
# ------------------------------------------------------------------ 优化器class Adam:    def __init__(self, P, lr=3e-3, b1=0.9, b2=0.999, eps=1e-8):        self.m = zeros_grad(P)        self.v = zeros_grad(P)        self.t = 0        self.lr, self.b1, self.b2, self.eps = lr, b1, b2, eps    def step(self, P, G, keys):        self.t += 1        for k in keys:            self.m[k] = self.b1 * self.m[k] + (1 - self.b1) * G[k]            self.v[k] = self.b2 * self.v[k] + (1 - self.b2) * np.square(G[k])            mh = self.m[k] / (1 - self.b1 ** self.t)            vh = self.v[k] / (1 - self.b2 ** self.t)            P[k] -= self.lr * mh / (np.sqrt(vh) + self.eps)class SGD:    """纯 SGD（无动量）。只用来做 B 组的负对照：它没有 Adam 的逐参数归一化，    α 会原封不动地放大步长，正好把「α 不变性到底是谁给的」切开。"""    def __init__(self, P, lr=1e-2):        self.lr = lr        self.acc = zeros_grad(P)          # 累计位移，用来核对「位移比 = α」    def step(self, P, G, keys):        for k in keys:            d = self.lr * G[k]            P[k] -= d            self.acc[k] = self.acc[k] + ddef clone(P):    return {k: v.copy() for k, v in P.items()}def acc(logits, y):    if logits.ndim == 3:        return float(np.mean(np.argmax(logits, axis=-1) == y))    return float(np.mean(np.argmax(logits, axis=-1) == y))

In [9]:
"""KI（knowledge insulation）companion lab —— 断在哪两个矩阵上，断了之后 α 还是不是旋钮。对应文章：π 系列具身模型（03）KI。设计原则：**只主张能被核到的事实**。论文层面的数字（7.5 倍步数、LIBERO 表、DROID 分数）是引用，本实验不重算；这里只负责四件能在 CPU 上钉死的事：  A 组  梯度通路的结构：为什么断 K_b / V_b 就够了，Q_b 和输出投影为什么不用管  B 组  α 的角色：断梯度之后 backbone 的训练轨迹与 α 完全无关（逐位）  C 组  语言跟随：几种配方各自拿什么换了什么  D 组  信息瓶颈：断梯度成立的前提是 backbone 还得被离散动作训练着纯 numpy / CPU / 固定 seed / 每组末尾断言 / 最后一个 cell 打出断言总数。反向是手写的（stop-gradient 在前向里是恒等映射，有限差分测不出它），再用有限差分在 sg 关闭时校准这份手写反向。一个必须说清的前提：本实验验的是**机制**（哪条梯度到得了哪个权重、代价是什么），不是论文的最终成绩。玩具 backbone 只有 32 维、K/V 瓶颈秩 8，三千步就能从零重新学会动作表示，所以「KI 收敛更快」这类规模相关的结论在这里体现不出来（§D 会给出实测的反向结果并说明原因）。"""SEED = 20261103N_TRAIN, N_EVAL = 2048, 512N_ASSERT = 0def ok(cond, msg):    global N_ASSERT    assert cond, f"FAILED: {msg}"    N_ASSERT += 1    print(f"  [ok] {msg}")def section(t):    print("\n" + "=" * 72)    print(t)    print("=" * 72)

## A 组：梯度通路的结构单批前向 + 反向就能验完，不需要训练。

In [10]:
# ================================================================== A 组section("A 组：梯度通路的结构（单批，无需训练）")rng = np.random.default_rng(SEED)sc = Scene(seed=4242, n=N_TRAIN + N_EVAL)PA = init_params(np.random.default_rng(1))bA = sample_batch(sc, np.arange(64), rng)out_n, cache_n = forward(PA, bA, sg=False)out_s, cache_s = forward(PA, bA, sg=True)M = MASKok(not M[:N_B, N_B:].any(), "掩码：backbone 的任何位置都看不见 expert token（P_ba = 0）")ok(not M[N_B:, N_O:N_B].any(), "掩码：expert 看不见 AR 动作 token 槽位（两类动作表示互不 attend）")ok(M[N_B:, :N_O].any() and M[:N_O, :N_O].all(),   "掩码：expert 看得见观测块，观测块内部双向")_, Gn, _, _ = grads(PA, bA, sg=False, w_flow=1.0, w_ar=0.0, w_lang=0.0)_, Gs, _, _ = grads(PA, bA, sg=True, w_flow=1.0, w_ar=0.0, w_lang=0.0)gb_k, gb_v = np.abs(Gn["Wk_b"]).max(), np.abs(Gn["Wv_b"]).max()ok(gb_k > 1e-4 and gb_v > 1e-4,   f"不断梯度时，流匹配损失确实会流进 backbone：|dWk_b|={gb_k:.3e}  |dWv_b|={gb_v:.3e}")ok(np.abs(Gn["Wq_b"]).max() == 0.0 and np.abs(Gn["Wo_b"]).max() == 0.0,   "不断梯度时 ∂L_flow/∂Wq_b 与 ∂L_flow/∂Wo_b 本来就是 0（backbone 行读不到 action token）")ok(np.abs(Gs["Wk_b"]).max() == 0.0 and np.abs(Gs["Wv_b"]).max() == 0.0,   "断梯度后 ∂L_flow/∂Wk_b 与 ∂L_flow/∂Wv_b 严格为 0（逐位）")ok(np.abs(Gs["Wq_a"]).max() > 1e-6,   f"断梯度后 expert 自己的 query 照常学习：|dWq_a|={np.abs(Gs['Wq_a']).max():.3e}")dmax = max(np.abs(out_n[k] - out_s[k]).max() for k in out_n)ok(dmax == 0.0, f"断梯度不改变前向：三个头输出的逐位最大差 = {dmax}")dmax_e = max(np.abs(Gs[k] - Gn[k]).max() for k in EXPERT)ok(dmax_e == 0.0, f"断梯度不改变 expert 侧梯度：逐位最大差 = {dmax_e}")_, Gar, _, _ = grads(PA, bA, sg=True, w_flow=0.0, w_ar=1.0, w_lang=0.0)ok(np.abs(Gar["Wk_b"]).max() > 1e-6 and np.abs(Gar["Wv_b"]).max() > 1e-6,   "断梯度后离散动作那条学习通路完好：∂L_AR/∂Wk_b、∂L_AR/∂Wv_b 非零")def loss_sum(P_, sg):    o, _ = forward(P_, bA, sg)    return (ce_loss(o["logits_lang"], bA["y_lang"])[0]            + ce_loss(o["logits_ar"], bA["y_ar"])[0]            + mse_loss(o["pred_flow"], bA["u"])[0])_, Gfd, _, _ = grads(PA, bA, sg=False, w_flow=1.0, w_ar=1.0, w_lang=1.0)eps, worst = 1e-6, 0.0for name in ALL_P:    W = PA[name]    for t in np.argsort(np.abs(Gfd[name]).ravel())[-4:]:        i = np.unravel_index(t, W.shape)        old = W[i]        W[i] = old + eps        lp = loss_sum(PA, False)        W[i] = old - eps        lm = loss_sum(PA, False)        W[i] = old        fd, an = (lp - lm) / (2 * eps), Gfd[name][i]        worst = max(worst, abs(fd - an) / max(abs(fd), abs(an), 1e-9))ok(worst < 1e-6, f"手写反向用有限差分校准（sg 关闭）：最大相对误差 {worst:.2e}")

========================================================================A 组：梯度通路的结构（单批，无需训练）========================================================================  [ok] 掩码：backbone 的任何位置都看不见 expert token（P_ba = 0）  [ok] 掩码：expert 看不见 AR 动作 token 槽位（两类动作表示互不 attend）  [ok] 掩码：expert 看得见观测块，观测块内部双向  [ok] 不断梯度时，流匹配损失确实会流进 backbone：|dWk_b|=5.524e-04  |dWv_b|=1.984e-02  [ok] 不断梯度时 ∂L_flow/∂Wq_b 与 ∂L_flow/∂Wo_b 本来就是 0（backbone 行读不到 action token）  [ok] 断梯度后 ∂L_flow/∂Wk_b 与 ∂L_flow/∂Wv_b 严格为 0（逐位）  [ok] 断梯度后 expert 自己的 query 照常学习：|dWq_a|=8.986e-03  [ok] 断梯度不改变前向：三个头输出的逐位最大差 = 0.0  [ok] 断梯度不改变 expert 侧梯度：逐位最大差 = 0.0  [ok] 断梯度后离散动作那条学习通路完好：∂L_AR/∂Wk_b、∂L_AR/∂Wv_b 非零  [ok] 手写反向用有限差分校准（sg 关闭）：最大相对误差 9.33e-07

## 训练：先预训练 backbone，再把随机初始化的 expert 嫁接上去六种配方都从同一份预训练权重出发，只有 expert 是随机初始化的——这正是 KI 论文要解决的问题场景。

In [11]:
# ================================================================== 训练section("训练：先预训练 backbone，再把随机初始化的 expert 嫁接上去")EVal = sample_batch(sc, np.arange(N_TRAIN, N_TRAIN + N_EVAL),                      np.random.default_rng(999), fixed_tau=0.5)# 「什么都不看」的下界：预测恒为 0（u 的均值就是 0）。评估一律钉在 τ=0.5，# 那里 u 与 x_t 严格独立（见 ki_lib 里 ω 的协方差构造），这条下界才是干净的。ACT_FLOOR = 0.25 * float(np.mean(np.square(EVal["u"])))print(f"  动作误差下界（预测恒 0）= {ACT_FLOOR:.4f}")# 负对照：把 4 个图像 token 换成别的样本的，语言任务应当掉到随机（1/4）EVal_shuf = dict(EVal)EVal_shuf["X_b"] = EVal["X_b"].copy()EVal_shuf["X_b"][:, :N_OBJ, :] = EVal["X_b"][    np.random.default_rng(4242).permutation(N_EVAL)][:, :N_OBJ, :]def metrics(P, batch=None):    b = EVal if batch is None else batch    o, _ = forward(P, b, sg=True)    return dict(act=float(np.mean(np.square(b["x_t"] - 0.5 * o["pred_flow"] - b["A"]))),                lang=acc(o["logits_lang"], b["y_lang"]),                ar=acc(o["logits_ar"], b["y_ar"]))def feats(P, idx):    """expert 真正能读到的东西：V_b = X_b @ Wv_b，注意力之前。"""    return (sc.x_b(idx) @ P["Wv_b"])[:, :N_O, :].mean(axis=1)def probe_r2(P, n_fit=1024, lam=1e-3):    Xf = np.concatenate([feats(P, np.arange(n_fit)), np.ones((n_fit, 1))], 1)    Xe = np.concatenate([feats(P, np.arange(N_TRAIN, N_TRAIN + N_EVAL)), np.ones((N_EVAL, 1))], 1)    Yf, Ye = sc.A[:n_fit].reshape(n_fit, -1), sc.A[N_TRAIN:].reshape(N_EVAL, -1)    W = np.linalg.solve(Xf.T @ Xf + lam * np.eye(Xf.shape[1]), Xf.T @ Yf)    return float(1 - np.sum((Xe @ W - Ye) ** 2) / np.sum((Ye - Ye.mean(0)) ** 2))def train(P, keys, sg, alpha, wa, wl, steps, lr, seed, pre=False):    r = np.random.default_rng(seed)    P = clone(P)    opt = Adam(P, lr=lr)    for _ in range(steps):        idx = r.integers(0, N_TRAIN, 64)        b = sample_batch(sc, idx, r)        _, G, _, _ = grads(P, b, sg, w_flow=alpha, w_ar=wa, w_lang=wl,                             ar_tgt=(b["y_ar_lang"] if pre else None))        opt.step(P, G, keys)    return P

========================================================================训练：先预训练 backbone，再把随机初始化的 expert 嫁接上去========================================================================  动作误差下界（预测恒 0）= 0.5152

### 预训练：只喂语言任务（对应 VLM 预训练）

In [12]:
PRE_LR, PRE_STEPS = 3e-3, 1200LR, STEPS = 5e-3, 3000P0 = init_params(np.random.default_rng(1))P0 = train(P0, BONE, False, 0.0, 1.0, 1.0, PRE_STEPS, PRE_LR, 7, pre=True)base = metrics(P0)R2_0 = probe_r2(P0)print(f"  预训练后：lang={base['lang']:.3f}  act={base['act']:.4f}  探针 R2={R2_0:.4f}")ok(base["lang"] > 0.70, f"预训练把语言任务做到了 {base['lang']:.3f}（后面所有配方都从这份权重出发）")ok(metrics(P0, EVal_shuf)["lang"] < 0.40,   f"负对照：打乱图像 token 后语言准确率掉到 {metrics(P0, EVal_shuf)['lang']:.3f}"   f"（随机水平 0.25）—— 说明这个任务真的在看图，不是背指令")

  预训练后：lang=0.756  act=0.5898  探针 R2=0.1636  [ok] 预训练把语言任务做到了 0.756（后面所有配方都从这份权重出发）  [ok] 负对照：打乱图像 token 后语言准确率掉到 0.268（随机水平 0.25）—— 说明这个任务真的在看图，不是背指令

### 六种配方 × α ∈ {1, 8, 32}`act` 是 τ=0.5 处的动作误差，下界（预测恒 0）约 0.515；`R2` 是从 backbone 的$$V_b$$ 特征线性预测动作块的决定系数，衡量「backbone 里到底有多少动作信息」。

In [13]:
RECIPES = dict(    naive=dict(sg=False, wa=0.0, wl=0.0, keys=ALL_P),       # π0：只有流匹配，不断梯度    joint=dict(sg=False, wa=1.0, wl=0.0, keys=ALL_P),       # 联合训练（离散 + 连续），不断梯度    joint_vlm=dict(sg=False, wa=1.0, wl=1.0, keys=ALL_P),    ki=dict(sg=True, wa=1.0, wl=0.0, keys=ALL_P),           # 断梯度 + 离散动作    ki_vlm=dict(sg=True, wa=1.0, wl=1.0, keys=ALL_P),       # 论文完整配方    freeze=dict(sg=True, wa=0.0, wl=0.0, keys=EXPERT),      # 冻住 backbone 只训 expert)RUN = {}for name, cfg in RECIPES.items():    for a in (1.0, 8.0, 32.0):        P = clone(init_params(np.random.default_rng(555)))        for k in BONE:            P[k] = P0[k].copy()                    # backbone 与离散头沿用预训练权重        P = train(P, cfg["keys"], cfg["sg"], a, cfg["wa"], cfg["wl"], STEPS, LR, 31)        RUN[(name, a)] = dict(P=P, m=metrics(P), r2=probe_r2(P))    print(f"  {name:10s} " + " | ".join(        f"a={a:<4g} lang={RUN[(name, a)]['m']['lang']:.3f} ar={RUN[(name, a)]['m']['ar']:.3f} "        f"act={RUN[(name, a)]['m']['act']:.4f} R2={RUN[(name, a)]['r2']:.3f}"        for a in (1.0, 8.0, 32.0)))def rel_diff(Pa, Pb, keys):    """平均逐元素差 / 平均逐元素幅值。"""    return max(float(np.abs(Pa[k] - Pb[k]).mean() / max(np.abs(Pb[k]).mean(), 1e-12))               for k in keys)

  naive      a=1    lang=0.359 ar=0.000 act=0.4236 R2=0.730 | a=8    lang=0.359 ar=0.000 act=0.4236 R2=0.730 | a=32   lang=0.359 ar=0.000 act=0.4236 R2=0.730  joint      a=1    lang=0.248 ar=0.381 act=0.4249 R2=0.732 | a=8    lang=0.219 ar=0.336 act=0.4238 R2=0.737 | a=32   lang=0.244 ar=0.333 act=0.4235 R2=0.737  joint_vlm  a=1    lang=0.805 ar=0.338 act=0.4390 R2=0.562 | a=8    lang=0.797 ar=0.329 act=0.4239 R2=0.733 | a=32   lang=0.789 ar=0.332 act=0.4239 R2=0.735  ki         a=1    lang=0.250 ar=0.370 act=0.4851 R2=0.340 | a=8    lang=0.250 ar=0.370 act=0.4851 R2=0.340 | a=32   lang=0.250 ar=0.370 act=0.4851 R2=0.340  ki_vlm     a=1    lang=0.795 ar=0.334 act=0.4794 R2=0.326 | a=8    lang=0.795 ar=0.334 act=0.4794 R2=0.326 | a=32   lang=0.795 ar=0.334 act=0.4794 R2=0.326  freeze     a=1    lang=0.756 ar=0.000 act=0.5190 R2=0.164 | a=8    lang=0.756 ar=0.000 act=0.5190 R2=0.164 | a=32   lang=0.756 ar=0.000 act=0.5190 R2=0.164

## B 组：α 还是不是旋钮

In [14]:
# ================================================================== B 组section("B 组：α 的角色 —— 断梯度之后它就不是旋钮了")for name in ("ki", "ki_vlm"):    dB = max(np.abs(RUN[(name, a)]["P"][k] - RUN[(name, 1.0)]["P"][k]).max()             for a in (8.0, 32.0) for k in BONE)    ok(dB == 0.0,       f"{name}：α = 1 / 8 / 32 三跑的 backbone 参数**逐位相同**（max 差 {dB:.1e}）")    dE = max(rel_diff(RUN[(name, a)]["P"], RUN[(name, 1.0)]["P"], EXPERT)             for a in (8.0, 32.0))    ok(dE < 1e-3, f"{name}：expert 参数也几乎不动（相对差 {dE:.1e}，残差只来自 Adam 的 ε）")for name in ("joint", "joint_vlm"):    d = max(rel_diff(RUN[(name, a)]["P"], RUN[(name, 1.0)]["P"], BONE)            for a in (8.0, 32.0))    ok(d > 0.05, f"{name}：不断梯度时 α 一变，backbone 轨迹就变（相对差 {d * 100:.0f}%）")j1, j8 = RUN[("joint_vlm", 1.0)], RUN[("joint_vlm", 8.0)]ok(abs(j8["r2"] - j1["r2"]) > 0.1,   f"joint_vlm：α 从 1 抬到 8，backbone 探针 R2 {j1['r2']:.3f} → {j8['r2']:.3f}"   f"（α 是真的旋钮，调大就是让流匹配梯度压过离散动作梯度）")ok(abs(RUN[("ki_vlm", 32.0)]["m"]["lang"] - RUN[("ki_vlm", 1.0)]["m"]["lang"]) < 1e-12,   "ki_vlm：语言跟随对 α 完全不敏感（逐位相同）")# 负对照：naive 只有一个损失，α 也不影响结果 —— 但机制不同：# 它的 backbone 梯度**带** α 这个因子，只是被 Adam 的逐参数归一化消掉了，# 所以只能做到「近似相同」，做不到 ki 那种「逐位相同」。dN = rel_diff(RUN[("naive", 32.0)]["P"], RUN[("naive", 1.0)]["P"], BONE)ok(0.0 < dN < 1e-2,   f"负对照 naive：单一损失下 α 也不改变结果（相对差 {dN:.1e}），"   f"但只是「近似」相同 —— 与 ki 的逐位为 0 不是一回事")# 把「梯度本身」和「优化器」切开：梯度仍然严格按 α 线性放大，# 是 Adam 的逐参数归一化把 α 消掉的（换成 SGD，步长就会原样放大 α 倍）。P_start = clone(init_params(np.random.default_rng(555)))for k in BONE:    P_start[k] = P0[k].copy()GB = {}for a in (1.0, 8.0, 32.0):    _, GB[a], _, _ = grads(P_start, bA, sg=True, w_flow=a, w_ar=0.0, w_lang=0.0)ok(all(np.abs(GB[a][k]).max() == 0.0 for a in GB for k in BONE),   "断梯度后，流匹配损失对 backbone 的梯度恒为 0 —— 与 α 无关，连「放大」都谈不上")worst = max(float(np.abs(GB[a][k] - a * GB[1.0][k]).max()                  / max(np.abs(a * GB[1.0][k]).max(), 1e-12))            for a in (8.0, 32.0) for k in EXPERT)ok(worst < 1e-12,   f"同一批数据上 expert 的梯度严格按 α 线性放大（相对误差 {worst:.1e}）——"   f"α 之所以不影响结果，是 Adam 逐参数归一化的功劳，不是梯度变小了")

========================================================================B 组：α 的角色 —— 断梯度之后它就不是旋钮了========================================================================  [ok] ki：α = 1 / 8 / 32 三跑的 backbone 参数**逐位相同**（max 差 0.0e+00）  [ok] ki：expert 参数也几乎不动（相对差 5.4e-06，残差只来自 Adam 的 ε）  [ok] ki_vlm：α = 1 / 8 / 32 三跑的 backbone 参数**逐位相同**（max 差 0.0e+00）  [ok] ki_vlm：expert 参数也几乎不动（相对差 3.7e-06，残差只来自 Adam 的 ε）  [ok] joint：不断梯度时 α 一变，backbone 轨迹就变（相对差 100%）  [ok] joint_vlm：不断梯度时 α 一变，backbone 轨迹就变（相对差 98%）  [ok] joint_vlm：α 从 1 抬到 8，backbone 探针 R2 0.562 → 0.733（α 是真的旋钮，调大就是让流匹配梯度压过离散动作梯度）  [ok] ki_vlm：语言跟随对 α 完全不敏感（逐位相同）  [ok] 负对照 naive：单一损失下 α 也不改变结果（相对差 4.0e-05），但只是「近似」相同 —— 与 ki 的逐位为 0 不是一回事  [ok] 断梯度后，流匹配损失对 backbone 的梯度恒为 0 —— 与 α 无关，连「放大」都谈不上  [ok] 同一批数据上 expert 的梯度严格按 α 线性放大（相对误差 0.0e+00）——α 之所以不影响结果，是 Adam 逐参数归一化的功劳，不是梯度变小了

## C 组：语言跟随 —— 各配方拿什么换了什么

In [15]:
# ================================================================== C 组section("C 组：语言跟随 —— 各配方拿什么换了什么")BL = base["lang"]for name in ("naive", "joint", "ki"):    ok(RUN[(name, 1.0)]["m"]["lang"] < BL - 0.15,       f"{name}（不带 VLM 协同训练）：语言跟随 {BL:.3f} → {RUN[(name, 1.0)]['m']['lang']:.3f}")for name in ("ki_vlm", "joint_vlm"):    ok(RUN[(name, 1.0)]["m"]["lang"] > BL - 0.05,       f"{name}（带 VLM 协同训练）：语言跟随保住在 {RUN[(name, 1.0)]['m']['lang']:.3f}")ok(abs(RUN[("freeze", 1.0)]["m"]["lang"] - BL) < 1e-12,   f"freeze：backbone 没动，语言跟随恒等于预训练值 {BL:.3f}")for name in ("joint", "joint_vlm", "ki", "ki_vlm"):    ok(RUN[(name, 1.0)]["m"]["ar"] > RUN[("naive", 1.0)]["m"]["ar"] + 0.2,       f"{name}：离散动作 token 准确率 {RUN[(name, 1.0)]['m']['ar']:.3f} "       f"远高于没训过它的 naive {RUN[('naive', 1.0)]['m']['ar']:.3f}")

========================================================================C 组：语言跟随 —— 各配方拿什么换了什么========================================================================  [ok] naive（不带 VLM 协同训练）：语言跟随 0.756 → 0.359  [ok] joint（不带 VLM 协同训练）：语言跟随 0.756 → 0.248  [ok] ki（不带 VLM 协同训练）：语言跟随 0.756 → 0.250  [ok] ki_vlm（带 VLM 协同训练）：语言跟随保住在 0.795  [ok] joint_vlm（带 VLM 协同训练）：语言跟随保住在 0.805  [ok] freeze：backbone 没动，语言跟随恒等于预训练值 0.756  [ok] joint：离散动作 token 准确率 0.381 远高于没训过它的 naive 0.000  [ok] joint_vlm：离散动作 token 准确率 0.338 远高于没训过它的 naive 0.000  [ok] ki：离散动作 token 准确率 0.370 远高于没训过它的 naive 0.000  [ok] ki_vlm：离散动作 token 准确率 0.334 远高于没训过它的 naive 0.000

## D 组：信息瓶颈 —— 断梯度成立的前提

In [16]:
# ================================================================== D 组section("D 组：信息瓶颈 —— 断梯度成立的前提")fr = RUN[("freeze", 1.0)]ok(fr["m"]["act"] > ACT_FLOOR - 0.01,   f"冻住 backbone：动作误差 {fr['m']['act']:.4f} 就卡在「什么都不看」的下界 "   f"{ACT_FLOOR:.4f} 上 —— 论文说的「冻住不行」在这里就是这样")ok(fr["r2"] < 0.25,   f"冻住的 backbone 里几乎没有动作信息：探针 R2 = {fr['r2']:.3f}"   f"（与预训练完的 {R2_0:.3f} 同量级，等于没多学）")ok(RUN[("ki_vlm", 1.0)]["r2"] > 1.5 * fr["r2"],   f"断梯度 + 离散动作训练后，backbone 学会承载动作信息：R2 {fr['r2']:.3f} → "   f"{RUN[('ki_vlm', 1.0)]['r2']:.3f}")ok(RUN[("ki_vlm", 1.0)]["m"]["act"] < fr["m"]["act"] - 0.02,   f"同一件事在动作误差上看：{fr['m']['act']:.4f} → {RUN[('ki_vlm', 1.0)]['m']['act']:.4f}")ok(RUN[("naive", 1.0)]["r2"] > 1.5 * RUN[("ki_vlm", 1.0)]["r2"],   f"代价：断掉梯度后动作信息只能从离散动作那条通路慢慢灌进 backbone，"   f"同样步数下探针 R2 {RUN[('naive', 1.0)]['r2']:.3f} vs {RUN[('ki_vlm', 1.0)]['r2']:.3f} "   f"（论文在真实规模上反而更快，玩具里 backbone 太小，体现不出来）")# 步数趋势：KI 到底追不追得上（玩具里追不上，这是规模问题不是机制问题）TREND = {}for tname, tsg, twa, twl in (("naive", False, 0.0, 0.0), ("ki_vlm", True, 1.0, 1.0)):    rows = []    for st in (1000, 3000, 9000):        P = clone(init_params(np.random.default_rng(555)))        for k in BONE:            P[k] = P0[k].copy()        P = train(P, ALL_P, tsg, 1.0, twa, twl, st, LR, 31)        rows.append((metrics(P), probe_r2(P)))    TREND[tname] = rows    print(f"  {tname:8s} " + " | ".join(        f"{st:5d} 步 lang={m['lang']:.3f} act={m['act']:.4f} R2={r:.3f}"        for st, (m, r) in zip((1000, 3000, 9000), rows)))ok(TREND["ki_vlm"][-1][1] < TREND["naive"][0][1],   f"步数趋势：KI 训到 9000 步（3 倍）主干探针 R2 {TREND['ki_vlm'][-1][1]:.3f} 仍低于 "   f"naive 1000 步的 {TREND['naive'][0][1]:.3f} —— 玩具里 KI 追不上，"   f"与论文「KI 收敛更快」方向相反（原因见文件头：玩具主干太小）")ok(TREND["ki_vlm"][1][0]["lang"] > TREND["naive"][1][0]["lang"] + 0.3,   f"但语言跟随的差距不随步数收窄：3000 步时 {TREND['naive'][1][0]['lang']:.3f} vs "   f"{TREND['ki_vlm'][1][0]['lang']:.3f}")# 负对照：动作只依赖当次实例，不依赖物体「名字」Xf = np.concatenate([sc.NAME[sc.c[:1024]], np.ones((1024, 1))], 1)Xe = np.concatenate([sc.NAME[sc.c[N_TRAIN:N_TRAIN + N_EVAL]], np.ones((N_EVAL, 1))], 1)Yf, Ye = sc.A[:1024].reshape(1024, -1), sc.A[N_TRAIN:].reshape(N_EVAL, -1)W = np.linalg.solve(Xf.T @ Xf + 1e-3 * np.eye(Xf.shape[1]), Xf.T @ Yf)r2_name = float(1 - np.sum((Xe @ W - Ye) ** 2) / np.sum((Ye - Ye.mean(0)) ** 2))ok(r2_name < 0.05,   f"负对照：光靠物体名字预测动作块的 R2 只有 {r2_name:.4f} —— 动作确实依赖当次实例，"   f"所以「VLM 预训练里没有机器人数据」这件事在探针里是真的")section(f"断言总数：{N_ASSERT}")print(f"断言总数 = {N_ASSERT}")

========================================================================D 组：信息瓶颈 —— 断梯度成立的前提========================================================================  [ok] 冻住 backbone：动作误差 0.5190 就卡在「什么都不看」的下界 0.5152 上 —— 论文说的「冻住不行」在这里就是这样  [ok] 冻住的 backbone 里几乎没有动作信息：探针 R2 = 0.164（与预训练完的 0.164 同量级，等于没多学）  [ok] 断梯度 + 离散动作训练后，backbone 学会承载动作信息：R2 0.164 → 0.326  [ok] 同一件事在动作误差上看：0.5190 → 0.4794  [ok] 代价：断掉梯度后动作信息只能从离散动作那条通路慢慢灌进 backbone，同样步数下探针 R2 0.730 vs 0.326 （论文在真实规模上反而更快，玩具里 backbone 太小，体现不出来）  naive     1000 步 lang=0.367 act=0.4278 R2=0.709 |  3000 步 lang=0.359 act=0.4236 R2=0.730 |  9000 步 lang=0.324 act=0.4245 R2=0.735  ki_vlm    1000 步 lang=0.775 act=0.5103 R2=0.250 |  3000 步 lang=0.795 act=0.4794 R2=0.326 |  9000 步 lang=0.781 act=0.4840 R2=0.281  [ok] 步数趋势：KI 训到 9000 步（3 倍）主干探针 R2 0.281 仍低于 naive 1000 步的 0.709 —— 玩具里 KI 追不上，与论文「KI 收敛更快」方向相反（原因见文件头：玩具主干太小）  [ok] 但语言跟随的差距不随步数收窄：3000 步时 0.359 vs 0.795  [ok] 负对照：光靠物体名字预测动作块的 R2 只有 -0.0075 —— 动作确实依赖当次实例，所以「VLM 预训练里没有机器人数据」这件事在探针里是真的=